# Stage1 日成交量参与度：冻结信息增量检验

完整 Stage1 原输入39项作为匹配对照，只新增一个“20日趋势为负时的成交量加权涨跌占比残差”。先核对 Qlib 来源、DatasetH 和标签成熟，不在结果出来后调整窗口、符号、模型或区间。2023+历史只作复用诊断，不能代表独立前向。

In [1]:
import torch
import json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd();card=root/'research/stage1_daily_volume_participation_information_20261022';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
source_spec=plan['candidate_source'];source_card=root/source_spec['card'];provider=root/source_spec['provider']
for file,key in [('qlib_store_manifest.json','provider_manifest_sha256')]:assert sha(provider/file)==source_spec[key]
for file,key in [('source_preflight_result.json','preflight_result_sha256'),('tail_bridge_result.json','tail_bridge_result_sha256')]:assert sha(source_card/file)==source_spec[key]
assert json.loads((source_card/'nas_source_archive_receipt.json').read_text())['archive_sha256']==source_spec['NAS_archive_sha256']
assert sha(root/plan['parent']['source_card']/'verification_manifest.json')==plan['parent']['verification_manifest_sha256']
parent_card=root/plan['parent']['source_card'];parent_manifest=json.loads((parent_card/'verification_manifest.json').read_text())
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:
    item=next(x for x in parent_manifest['entries'] if x['path'].endswith('/artifacts/'+name))
    assert item['sha256']==plan['parent'][key] and sha(parent_card/item['path'])==item['sha256']
qlib.init(provider_uri=str(provider),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time='2026-10-07',freq='day'))
price=D.features(['SOXX'],['$close','$volume'],start_time='2010-03-11',end_time='2026-10-07',freq='day').xs('SOXX',level='instrument').sort_index()
assert price.index.equals(calendar)
with R.uri_context(str(parent_card/'qlib_recorder')):
    recorder=R.get_recorder(recorder_id=plan['parent']['recorder_id'],experiment_id='905520356740140469')
    original=recorder.load_object('models/base_outer_71')
    base=recorder.load_object('inputs/features')
    frame=recorder.load_object('inputs/daily_frame')
    timing=recorder.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
ret=price['$close'].pct_change();sign=np.sign(ret);volume=price['$volume'].astype(float)
residual=((sign*volume).rolling(20,min_periods=20).sum()/volume.rolling(20,min_periods=20).sum()-sign.rolling(20,min_periods=20).mean()).replace([np.inf,-np.inf],np.nan)
feature=residual.reindex(frame.index).where(base.soxx_ret20.lt(0),0.0).rename('volume_participation_in_negative_trend')
forward=frame['soxx_forward_open_return'].astype(float)
parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert feature.loc[train_dates].notna().all() and feature.loc[held_dates].notna().all()
assert train_dates.sum()>=plan['cohorts']['minimum_mature_train_rows'] and held_dates.sum()>=plan['cohorts']['minimum_mature_held_rows']
augmented=base[base_names].copy();augmented[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
train_prepared=dataset.prepare('train',col_set=['feature','label','meta'])
held_prepared=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'source_and_DatasetH_maturity_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'parent_recorder_id':plan['parent']['recorder_id'],'source_provider_manifest_sha256':source_spec['provider_manifest_sha256'],'source_nas_archive_sha256':source_spec['NAS_archive_sha256'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'training_last_label_available_at':str(mature.loc[train_dates].max()),'held_last_label_available_at':str(mature.loc[held_dates].max()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_nonzero_rows':int(feature.loc[train_dates].ne(0).sum()),'candidate_held_nonzero_rows':int(feature.loc[held_dates].ne(0).sum()),'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False}
(card/'preflight_result.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print({k:report[k] for k in ['status','training_mature_rows','held_mature_rows','candidate_train_nonzero_rows','candidate_held_nonzero_rows']})

{'status': 'source_and_DatasetH_maturity_preflight_passed_no_fit', 'training_mature_rows': 3014, 'held_mature_rows': 940, 'candidate_train_nonzero_rows': 1132, 'candidate_held_nonzero_rows': 357}


## 中文结论

本预检只核对来源与标签成熟；计划固定的两次 Ridge 拟合和一次区块检验必须按尝试记录执行。若信息门失败，关闭固定特征；若通过，也仅能另冻结 Stage1 0bp 全账户实验。